In [11]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 01_cheque_bounce_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 02_rent_arrears_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 03_consumer_product_defect_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 04_property_payment_demand_notice_sample.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

In [13]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [14]:

chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 8 chunks

Example chunk:
Content: SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE
 LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE
 INSTRUMENTS ACT, 1881
Notice No.
LL-TEST-2026-001
Date
14 August 2026
To
Mr. Rohan Mehta
Address
24,...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

Embeddings and vectorDB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4171.58it/s]


Model loaded successfully. Embedding dimension: 384


In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [18]:
chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

In [ ]:
### Convert the text to embeddings
####chunks = complete Document objects
##texts = only the actual text extracted from those objects

texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]

Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 8


RAG Retriever Pipeline
